## 1. Problem and learning goals

A shop wants to rank four products by revenue while keeping every product matched to its original revenue value.

**Tasks from the slides**
1. Create product-name and revenue arrays.
2. Sort revenue in ascending and descending order.
3. Use `np.argsort()` to rank the **products**, not just the revenue values.
4. Report the top two products and their revenues.

**Deliverable:** A correct leaderboard and an explanation of why **shared sorting indices** are essential.

## 2. Create aligned product and revenue arrays

Each index identifies one record. For example, `products[1]` is `"Book"`, and `revenue[1]` is `350`.

We use the **same values as the teaching slide** and interpret revenue as pounds sterling (£).

In [1]:
import numpy as np

products = np.array(["Pen", "Book", "Bag", "Mug"])
revenue = np.array([120, 350, 280, 90])

print("Original records:")
for name, amount in zip(products, revenue):
    print(f"  {name:<5} £{amount:>3}")

assert products.shape == revenue.shape == (4,)

Original records:
  Pen   £120
  Book  £350
  Bag   £280
  Mug   £ 90


## 3. Task 2 — Sort revenue ascending and descending

`np.sort(revenue)` returns a **new sorted array** in ascending order, without changing the original `revenue` array.

For descending order, reverse the sorted result with `[::-1]`.

**Expected values:** Ascending `[90, 120, 280, 350]`; descending `[350, 280, 120, 90]`.

In [2]:
ascending = np.sort(revenue)
descending = np.sort(revenue)[::-1]

print("Ascending revenue:", ascending)
print("Descending revenue:", descending)
print("Original revenue unchanged:", revenue)

np.testing.assert_array_equal(ascending, np.array([90, 120, 280, 350]))
np.testing.assert_array_equal(descending, np.array([350, 280, 120, 90]))

Ascending revenue: [ 90 120 280 350]
Descending revenue: [350 280 120  90]
Original revenue unchanged: [120 350 280  90]


## 4. Why sorting revenues alone is not enough

If we sort `revenue` but leave `products` in the original order, the **product–revenue pairs become incorrect**. For example, pairing `products[0]` (Pen) with the largest sorted revenue (350) would falsely claim Pen earned £350.

The cell below **deliberately demonstrates the incorrect approach**. Do not use this output as a real leaderboard.

In [3]:
print("INCORRECT pairing — shown only to explain the mistake:")
for name, amount in zip(products, descending):
    print(f"  {name:<5} £{amount:>3}  <-- potentially mismatched")

INCORRECT pairing — shown only to explain the mistake:
  Pen   £350  <-- potentially mismatched
  Book  £280  <-- potentially mismatched
  Bag   £120  <-- potentially mismatched
  Mug   £ 90  <-- potentially mismatched


## 5. Task 3 — Use `np.argsort()` to keep records aligned

`np.argsort(revenue)` returns the **indices that would sort** revenue in ascending order. Reversing these indices gives descending order.

We can apply the **same indices** to both arrays:

```python
order = np.argsort(revenue)[::-1]
products[order]   # product names in descending revenue order
revenue[order]    # matching revenue values
```

In [4]:
order = np.argsort(revenue)[::-1]
ranked_products = products[order]
ranked_revenue = revenue[order]

print("Indices, highest revenue first:", order)
print("Ranked product names:", ranked_products)
print("Ranked revenue values:", ranked_revenue)

np.testing.assert_array_equal(order, np.array([1, 2, 0, 3]))

Indices, highest revenue first: [1 2 0 3]
Ranked product names: ['Book' 'Bag' 'Pen' 'Mug']
Ranked revenue values: [350 280 120  90]


## 6. Display the complete sales leaderboard

We use the index order to display each name alongside its matching revenue. The ranking is **Book, Bag, Pen, Mug**.

In [5]:
print("SALES LEADERBOARD")
print("-----------------")
for rank, idx in enumerate(order, start=1):
    print(f"{rank}. {products[idx]:<5} £{revenue[idx]:.2f}")

SALES LEADERBOARD
-----------------
1. Book  £350.00
2. Bag   £280.00
3. Pen   £120.00
4. Mug   £90.00


## 7. Task 4 — Report the top two products

`order[:2]` selects the first two indices from the descending ranking. The expected top two are:

1. **Book — £350**
2. **Bag — £280**

In [6]:
top_two_indices = order[:2]
print("TOP TWO PRODUCTS")
for idx in top_two_indices:
    print(f"{products[idx]}: £{revenue[idx]:.2f}")

np.testing.assert_array_equal(products[top_two_indices], np.array(["Book", "Bag"]))
np.testing.assert_array_equal(revenue[top_two_indices], np.array([350, 280]))

TOP TWO PRODUCTS
Book: £350.00
Bag: £280.00


## 8. Validate that no record was lost or mismatched

Ranking must change only the **order**, not the number of records, the total revenue or the association between each product and its revenue.

In [7]:
assert len(order) == len(products) == len(revenue)
assert len(set(order.tolist())) == len(order)  # no repeated index
assert revenue.sum() == ranked_revenue.sum()
assert all(revenue[idx] == ranked_revenue[rank] for rank, idx in enumerate(order))

print("All checks passed.")
print(f"Total revenue across four products: £{revenue.sum():.2f}")

All checks passed.
Total revenue across four products: £840.00


## 9. Additional demonstration — Equal revenues (ties)

When two products have equal revenue, a ranking needs a **tie policy**. A stable sort preserves the original relative order of tied records.

`np.argsort(-values, kind="stable")` sorts descending while preserving the original order among ties. This is an **optional extension** beyond the four activity tasks.

In [8]:
tie_products = np.array(["A", "B", "C", "D"])
tie_revenue = np.array([200, 350, 350, 100])
stable_order = np.argsort(-tie_revenue, kind="stable")

print("Leaderboard with a tie:")
for idx in stable_order:
    print(f"  {tie_products[idx]}: £{tie_revenue[idx]}")

# B precedes C because they are tied and B appeared first originally.
np.testing.assert_array_equal(tie_products[stable_order[:2]], ["B", "C"])

Leaderboard with a tie:
  B: £350
  C: £350
  A: £200
  D: £100


## 10. Expected results and interpretation

| Result | Expected value |
|---|---|
| Original revenue | `[120, 350, 280, 90]` |
| Ascending | `[90, 120, 280, 350]` |
| Descending | `[350, 280, 120, 90]` |
| Descending indices | `[1, 2, 0, 3]` |
| Full leaderboard | Book, Bag, Pen, Mug |
| **Top two** | **Book (£350), Bag (£280)** |

**Common mistakes:** Sorting product names and revenues independently; assuming `np.sort` changes the original array; confusing `argsort` (indices) with `sort` (values); or forgetting to decide how missing values and ties should be handled.

## 11. Review questions (with model answers)

**Q1. Does `np.sort(a)` modify `a`?** No. It returns a sorted copy; `a.sort()` sorts the array in place.

**Q2. What does `np.argsort()` return?** The indices that would sort the array.

**Q3. Why apply the same index order to both arrays?** To keep each product paired with its original revenue.

**Q4. How should ties or `NaN` values be handled?** Define a consistent policy before ranking. A stable sort can preserve the input order for ties; missing values should be handled explicitly according to the analysis objective.

**Optional extension:** Calculate the percentage of total revenue contributed by the top two products. Would the leaderboard change if every revenue increased by 10%?